# Colab 36 — does the length constraint have to exist?

Every notebook since `colab11` has filtered the CATH pool to sequence length **[50, 200]** and padded the
encoder input to a fixed width of 200. Those are not two decisions, they are one, and this notebook
separates them.

### The mechanism being tested

`encode_pad` pads to `MAX_LEN`, the mask zeroes activations at pad positions, and *then*
`AdaptiveAvgPool1d(K)` splits the **fixed padded axis** into K buckets. Bucket boundaries therefore sit at
fixed absolute positions, not at fractions of the sequence. At `MAX_LEN = 200` a median CATH domain (~110
residues) fills roughly 9 of 16 buckets and the remaining 7 are **structurally zero** — which is very
likely the mechanism behind the measured `UMAP-1 ~ length` correlation (rho = -0.955, colab23).

Naively raising `MAX_LEN` to the pool maximum (1202) makes each bucket ~75 positions wide, so that same
domain would fill 1-2 of 16 buckets. The model would not fail because long sequences are hard; it would
fail because the representation of everything short was destroyed.

**True-length pooling** fixes this by applying *PyTorch's own* `AdaptiveAvgPool1d` rule to each
sequence's real length `L`: bucket `b` covers `[floor(b*L/K), ceil((b+1)*L/K))`. Note these windows
**overlap** when `L` is not divisible by `K`, and repeat positions when `L < K` — that is what
`AdaptiveAvgPool1d` does, and reproducing it exactly is deliberate: it means A0 -> A2 changes only the
*coordinate frame*, not the pooling rule. Section 2 asserts equality with `F.adaptive_avg_pool1d` when
the tensor carries no padding.

### Registered predictions (write down before running, so the run can refute them)

| Quantity | Prediction |
|---|---|
| 3Di queries@0.70, unrestricted | 347 -> ~1000+ (66.7% of 3Di high-sim space is below length 50) |
| SS queries | 10,002 -> ~14,000; already near-saturated |
| AA | stays ~5 pairs — a redundancy-reduction problem, not a length problem (strict: 4 pairs / 8 queries) |
| 3Di MAP@10 | **falls** (0.515 -> ~0.35-0.45) while queries triple: new positives are short-short pairs |
| Score distributions | all medians shift left as extreme length ratios pile into the far band |
| **A1** (fixed pooling, full width) | large degradation, worst in the high band. If it does *not* degrade, the zero-bucket mechanism is wrong |
| **A2** (true-length pooling) | may be **worse** than A0 on retrieval despite being cleaner: normLev is length-dependent by construction, and true-length pooling discards the length cue |
| **A3/A4** (K=32) | small; concentrated on long sequences. **A4 is a matched-parameter comparison, not a clean isolation of resolution** — it changes K *and* conv2 width together, so it bounds the capacity explanation rather than removing it |
| **A5** (train 20-800) | matters more than K for long sequences |

### The trap this notebook is built to avoid

`normLev <= min(|a|,|b|) / max(|a|,|b|)` — length ratio is a **hard ceiling** on similarity before a single
character is compared. In [50, 200] the worst ratio is 4:1; unrestricted it is 130:1. So length alone
becomes a much better predictor of normLev, and a metric that improves may be measuring length-matching
rather than edit distance. Hence the **length-ratio baseline** (`Length` in every table): it scores pairs by
`min(L)/max(L)` and nothing else. Read every SNNEED number against it.

### Arms

| Arm | Pooling | Pad width | K | conv2 ch | Train lengths | Params |
|---|---|---|---|---|---|---|
| **A0 deployed (reference)** | padded-width | 200 (truncates >200) | 16 | 64 | 50-200 | 141,184 |
| **A1 fixed-pool, full width** | padded-width | pool max | 16 | 64 | 50-200 | 141,184 |
| **A2 true-length K16** | true-length | n/a | 16 | 64 | 50-200 | 141,184 |
| **A3 true-length K32** | true-length | n/a | 32 | 64 | 50-200 | 272,256 |
| **A4 true-length K32, matched-parameter** | true-length | n/a | 32 | 32 | 50-200 | 138,080 |
| **A5 true-length K16, wide train** | true-length | n/a | 16 | 64 | 20-800 | 141,184 |

**A0 on the `deployed` variant is a hard gate.** It is trained and evaluated *first*, and the run aborts
unless it reproduces `colab35_metrics.csv` within tolerance (Spearman 0.926 / 0.953 / 0.963 / 0.183,
MAP@10 0.972 / 0.515 / 0.405 / 0.928). The pool audit in section 6 is likewise an assertion, not a print.

### Pool variants — four, because the current filter is not what it looks like

| Variant | Rule | Pool (AA / SS / 3Di) |
|---|---|---|
| `none` | no length filter at all | 14,907 / 14,907 / 14,907 |
| `min20` | length >= 20 | ~14,868 |
| **`deployed`** | `50 <= L <= 200` **OR** domain in `RESCUED` | **10,501 / 10,497 / 10,501** |
| `strict` | `50 <= L <= 200` | 10,499 / 10,495 / 10,499 |

`RESCUED = {'4z0mC02', '3qkaE02'}` (lengths **34** and **43**) is the outcome-aware exception that every
notebook since colab11 has carried. **`deployed` is the pool colab35 actually used**, so it — not
`strict` — is the replication reference.

The exception is not merely two extra sequences. Measured on the committed data:

- their mutual pair has **AA normLev 0.744** and **SS normLev 0.791**, so it enters the >= 0.70 oracle;
- it is **1 of AA's 5 high-similarity pairs** — strict AA has 4 pairs / 8 queries;
- on SS, `3qkaE02` (length 43, 3-letter alphabet) has **~200 partners at >= 0.70**, because short strings
  over a 3-letter alphabet clear 0.70 by chance. It materially inflates the SS query set.

`deployed` vs `strict` therefore measures the cost of the outcome-aware filter directly, which is what
`CONTINUE_v22` open item 5.3 has been waiting for.

**Note:** under `none` and `min20` there is no length rule for the exception to be an exception *to* —
both domains enter under the general criterion, and the filter dissolves.

## 1. Setup

In [ ]:
import os
os.chdir('/content')
!rm -rf thesis-edit-distance-nn
!git clone https://github.com/katzemelli/thesis-edit-distance-nn.git
os.chdir('/content/thesis-edit-distance-nn')

In [ ]:
DATA_DIR = '/content/thesis-edit-distance-nn/sampledata/cath'
for f in ['cath_s20_train70.csv.gz', 'cath_s20_test30.csv.gz', 'cath_s20_3di.csv.gz']:
    p = os.path.join(DATA_DIR, f); print(f'{"OK" if os.path.exists(p) else "MISSING":<8} {p}')

In [ ]:
!pip install torch rapidfuzz scikit-learn scipy matplotlib --quiet

In [ ]:
# Optional: persist the oracle to Drive so a runtime disconnect does not cost 45 minutes.
# Set USE_DRIVE = True and run; leave False to cache in the ephemeral Colab filesystem only.
USE_DRIVE  = False
DRIVE_DIR  = '/content/drive/MyDrive/thesis_cache/colab36'
if USE_DRIVE:
    from google.colab import drive; drive.mount('/content/drive')
    os.makedirs(DRIVE_DIR, exist_ok=True)
CACHE_DIR = DRIVE_DIR if USE_DRIVE else '/content/thesis-edit-distance-nn/_cache36'
os.makedirs(CACHE_DIR, exist_ok=True)
print('oracle cache ->', CACHE_DIR)

In [ ]:
import json, platform, subprocess, sys, datetime

def _v(mod):
    try: return __import__(mod).__version__
    except Exception as e: return f'<unavailable: {e}>'

ENV = {'captured_utc': datetime.datetime.utcnow().isoformat(timespec='seconds') + 'Z',
       'python': sys.version.split()[0], 'platform': platform.platform()}
for m in ['torch', 'numpy', 'pandas', 'scipy', 'sklearn', 'rapidfuzz', 'matplotlib']:
    ENV[m] = _v(m)
try:
    import torch as _t
    ENV['cuda_available'] = _t.cuda.is_available()
    ENV['cuda_device'] = _t.cuda.get_device_name(0) if _t.cuda.is_available() else None
    ENV['cuda_version'] = _t.version.cuda
except Exception: pass
try: ENV['git_commit'] = subprocess.check_output(['git','rev-parse','HEAD'], text=True).strip()
except Exception: ENV['git_commit'] = '<unknown>'

# METHODS TODO — still unrecorded anywhere in the repo. Fill in and keep.
ENV['cath_release'] = 'TODO - exact CATH release, S20 file name, download date'
ENV['3di_source']   = 'TODO - Foldseek version used to generate the 3Di strings'

with open('environment_colab36.json','w') as fh: json.dump(ENV, fh, indent=2)
for k, v in ENV.items(): print(f'{k:<16} {v}')

In [ ]:
import time, numpy as np, pandas as pd, torch
import torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from scipy.stats import spearmanr
from sklearn.metrics import roc_auc_score
from rapidfuzz.distance import Levenshtein as RFLev
from rapidfuzz.process import cdist as rf_cdist
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

QUICK   = False                    # True -> 1 seed, smaller samples; ~15 min end to end
N_TRAIN = 30_000
SEEDS   = [0] if QUICK else [0, 1, 2]
EPOCHS  = 30
STRAT_PER_BIN, STRAT_CAND = 400, (40_000 if QUICK else 200_000)
NDIST                     = 30_000 if QUICK else 150_000   # pairs for the score-distribution figure
SYN_PERTURB, SYN_INDEP    = (4_000, 1_600) if QUICK else (20_000, 8_000)
FEED_ORDER   = ['synth', '3Di', 'SS', 'AA']
VARIANTS     = ['none', 'min20', 'deployed', 'strict']
PRIMARY_ARM  = 'A2 true K16'   # designated in advance; best-of-six is exploratory only
print(f'seeds={SEEDS} epochs={EPOCHS} n_train={N_TRAIN} quick={QUICK}')

## 2. The encoder, with pooling as a switch

`pool_mode='fixed'` is the deployed behaviour: `AdaptiveAvgPool1d(K)` over the whole padded axis, so bucket
boundaries are absolute and depend on `pad_w`. `pool_mode='true'` pools over each sequence's real length,
so bucket `b` always covers the same *fraction* of the sequence.

Everything else is unchanged from `colab35`: plain unweighted MSE on `normLev` through the parameter-free
readout `1 - ||e_a - e_b||_2 / 2`. No head, no bins, no loss weights.

In [ ]:
AA_ALPHABET = 'ACDEFGHIKLMNPQRSTVWY'; SS_ALPHABET = 'HLS'
CHAR_TO_IDX = {c: i for i, c in enumerate(AA_ALPHABET)}; PAD_IDX = 20; VOCAB = 21
BS = 128
BAND_LOW, BAND_HIGH = 0.30, 0.70
AA_SET, SS_SET = set(AA_ALPHABET), set(SS_ALPHABET)
is_aa = lambda s: all(c in AA_SET for c in s); is_ss = lambda s: all(c in SS_SET for c in s)

def norm_lev(a, b):
    L = max(len(a), len(b)); return 1.0 if L == 0 else 1.0 - RFLev.distance(a, b) / L

def encode_batch(seqs, width, truncate):
    """Pad (and optionally truncate) a list of strings to `width`. Returns (idx, lengths)."""
    out = np.full((len(seqs), width), PAD_IDX, dtype=np.int64); lens = np.zeros(len(seqs), dtype=np.int64)
    for r, s in enumerate(seqs):
        if truncate: s = s[:width]
        if len(s) > width:
            raise ValueError(f'sequence of length {len(s)} exceeds pad width {width} with truncate=False')
        out[r, :len(s)] = [CHAR_TO_IDX[c] for c in s]; lens[r] = len(s)
    return torch.from_numpy(out), torch.from_numpy(lens)

def perturb(seq, k, abc, rng, cap):
    s = list(seq); abc = list(abc)
    for _ in range(k):
        if len(s) == 0: op = 'ins'
        elif len(s) >= cap: op = rng.choice(['sub','del'])
        else: op = rng.choice(['sub','ins','del'])
        if op == 'sub': i = rng.integers(0, len(s)); s[i] = rng.choice([c for c in abc if c != s[i]])
        elif op == 'ins': i = rng.integers(0, len(s)+1); s.insert(i, rng.choice(abc))
        else: i = rng.integers(0, len(s)); del s[i]
    return ''.join(s)

def rand_seq(abc, rng, lo, hi):
    L = int(rng.integers(lo, hi+1)); return ''.join(rng.choice(list(abc), size=L))

In [ ]:
def pool_true(h, lengths, K):
    """AdaptiveAvgPool1d(K) semantics applied to each sequence's REAL length.

    PyTorch's adaptive pooling uses bucket b = [floor(b*L/K), ceil((b+1)*L/K)), which OVERLAP when L is
    not divisible by K (and repeat elements when L < K). We reproduce that exactly, so the only thing
    that changes between A0 (pool over the padded width) and A2 (pool over the true length) is the
    coordinate frame -- not the pooling rule. Verified against F.adaptive_avg_pool1d in section 2.
    """
    B, Cc, W = h.shape
    L = lengths.to(h.device).clamp(min=1).to(h.dtype).unsqueeze(1)              # B,1
    b = torch.arange(K, device=h.device, dtype=h.dtype).unsqueeze(0)            # 1,K
    start = torch.floor(b * L / K); end = torch.ceil((b + 1) * L / K)           # B,K
    pos = torch.arange(W, device=h.device, dtype=h.dtype).view(1, 1, W)
    m = ((pos >= start.unsqueeze(2)) & (pos < end.unsqueeze(2))).to(h.dtype)    # B,K,W
    m = m / m.sum(2, keepdim=True).clamp(min=1.0)
    return torch.bmm(h, m.transpose(1, 2))                                      # B,C,K

class EncPool(nn.Module):
    def __init__(s, K=16, c2ch=64, pool_mode='fixed'):
        super().__init__()
        s.K, s.pool_mode = K, pool_mode
        s.emb = nn.Embedding(VOCAB, 32, padding_idx=PAD_IDX)
        s.c1  = nn.Conv1d(32, 32, 3, padding=1); s.c2 = nn.Conv1d(32, c2ch, 3, padding=1)
        s.fc  = nn.Linear(c2ch * K, 128)
    def forward(s, x, lengths):
        m = (x != PAD_IDX).float(); e = s.emb(x).permute(0, 2, 1)
        h = F.relu(s.c1(e)); h = F.relu(s.c2(h)); h = h * m.unsqueeze(1)
        p = F.adaptive_avg_pool1d(h, s.K) if s.pool_mode == 'fixed' else pool_true(h, lengths, s.K)
        return F.normalize(s.fc(p.flatten(1)), p=2, dim=1)

class RegModel(nn.Module):
    def __init__(s, enc): super().__init__(); s.encoder = enc
    def forward(s, xa, la, xb, lb):
        ea, eb = s.encoder(xa, la), s.encoder(xb, lb)
        return 1.0 - torch.linalg.vector_norm(ea - eb, ord=2, dim=1) / 2.0

In [ ]:
# ---- arm table -------------------------------------------------------------
# pad_w = None  -> pad each batch to its own maximum (identity for true-length pooling)
# pad_w = 200   -> deployed behaviour, truncates anything longer
ARMS = [
    dict(name='A0 deployed',        pool='fixed', K=16, c2ch=64, pad_w=200,  truncate=True,  train_len=(50, 200)),
    dict(name='A1 fixed fullwidth', pool='fixed', K=16, c2ch=64, pad_w='max', truncate=False, train_len=(50, 200)),
    dict(name='A2 true K16',        pool='true',  K=16, c2ch=64, pad_w=None, truncate=False, train_len=(50, 200)),
    dict(name='A3 true K32',        pool='true',  K=32, c2ch=64, pad_w=None, truncate=False, train_len=(50, 200)),
    dict(name='A4 true K32 matched',pool='true',  K=32, c2ch=32, pad_w=None, truncate=False, train_len=(50, 200)),
    dict(name='A5 true K16 wide',   pool='true',  K=16, c2ch=64, pad_w=None, truncate=False, train_len=(20, 800)),
]
for a in ARMS:
    n = sum(p.numel() for p in EncPool(a['K'], a['c2ch'], a['pool']).parameters())
    a['params'] = n
    print(f"{a['name']:<22} pool={a['pool']:<5} K={a['K']:<3} c2ch={a['c2ch']:<3} "
          f"train_len={a['train_len']}  params={n:,}")
print('\nA0 must be 141,184 (matches colab35). A4 must be below A0 — that is the capacity control.')

# Invariance check: with true-length pooling, an embedding must not depend on how much padding
# it happened to be batched with. Requires >= 2 pad positions (conv1d padding=1, applied twice).
_e = EncPool(16, 64, 'true').eval(); _s = ('ACDEFGHIKLMNPQRSTVWY' * 5)[:83]
with torch.no_grad():
    _a, _la = encode_batch([_s], len(_s) + 2, False); _b, _lb = encode_batch([_s], len(_s) + 400, False)
    _d = (_e(_a, _la) - _e(_b, _lb)).abs().max().item()
print(f'true-length pooling batch-invariance: max |delta| = {_d:.2e}  '
      f'{"OK" if _d < 1e-5 else "*** FAIL — do not trust the true-length arms"}')

# pool_true must equal PyTorch adaptive pooling when the tensor has no padding, otherwise A0 -> A2
# would change the pooling RULE as well as the coordinate frame.
_h = torch.randn(4, 5, 37); _L = torch.tensor([37, 37, 37, 37])
for _K in (3, 16, 32):
    _delta = (pool_true(_h, _L, _K) - F.adaptive_avg_pool1d(_h, _K)).abs().max().item()
    print(f'  pool_true == adaptive_avg_pool1d at L=W, K={_K:<3}: max |delta| = {_delta:.2e} '
          f'{"OK" if _delta < 1e-6 else "*** FAIL"}')

## 3. Pools — one unrestricted pool per feed, three length variants as index subsets

The unrestricted pool is built once. `min20` and `50-200` are **index subsets** of it, which is what makes
the single oracle build in section 4 valid: `normLev(a, b)` does not depend on which pool a and b sit in.

In [ ]:
raw = pd.concat([pd.read_csv(f'{DATA_DIR}/cath_s20_train70.csv.gz'),
                 pd.read_csv(f'{DATA_DIR}/cath_s20_test30.csv.gz')],
                ignore_index=True).drop_duplicates('domain_id')
seqs3 = pd.read_csv(f'{DATA_DIR}/cath_s20_3di.csv.gz')

# No length rule here — that is the experiment. Only the alphabet criterion applies.
# NOTE: with no length filter, RESCUED = {'4z0mC02','3qkaE02'} has no exception to be:
# both domains enter under the general criterion. See the header.
RESCUED = {'4z0mC02', '3qkaE02'}      # lengths 34 and 43 — see the variant table below

def _collect(df, idcol, seqcol, ok):
    d = {}
    for i, s in zip(df[idcol], df[seqcol].astype(str)):
        if isinstance(s, str) and len(s) > 0 and ok(s): d[i] = s
    return d

LOOK = {'AA':  _collect(raw,   'domain_id', 'aa_seq', is_aa),
        'SS':  _collect(raw,   'domain_id', 'ss_seq', is_ss),
        '3Di': _collect(seqs3, 'domain_id', '3di',    is_aa)}
CATH_FEEDS = ['3Di', 'SS', 'AA']
POOL_SEQ = {f: list(LOOK[f].values()) for f in CATH_FEEDS}
POOL_ID  = {f: list(LOOK[f].keys())   for f in CATH_FEEDS}
POOL_LEN = {f: np.array([len(s) for s in POOL_SEQ[f]]) for f in CATH_FEEDS}

if QUICK:      # smoke pass only — subsample so the exhaustive oracle is ~50x cheaper
    _r = np.random.default_rng(0)
    for f in CATH_FEEDS:
        sel = np.sort(_r.permutation(len(POOL_SEQ[f]))[:2000])
        keep_r = [i for i, d in enumerate(POOL_ID[f]) if d in RESCUED]
        sel = np.unique(np.concatenate([sel, np.array(keep_r, dtype=int)]))
        POOL_SEQ[f] = [POOL_SEQ[f][i] for i in sel]; POOL_ID[f] = [POOL_ID[f][i] for i in sel]
        POOL_LEN[f] = POOL_LEN[f][sel]
    print('*** QUICK: pools subsampled to ~2000 sequences. Numbers are NOT results. ***\n')

# 'deployed' reproduces colab35 exactly: [50,200] OR one of the two RESCUED domains.
# 'strict'   is the same filter with the outcome-aware exception removed.
def _var_idx(f, v):
    L = POOL_LEN[f]; ids = POOL_ID[f]
    if v == 'none':   m = L >= 1
    elif v == 'min20': m = L >= 20
    else:
        m = (L >= 50) & (L <= 200)
        if v == 'deployed':
            m = m | np.array([d in RESCUED for d in ids])
    return np.where(m)[0]
VAR_IDX = {f: {v: _var_idx(f, v) for v in VARIANTS} for f in CATH_FEEDS}

PAD_MAX = int(max(POOL_LEN[f].max() for f in CATH_FEEDS))
print(f'global pad width (pool maximum) = {PAD_MAX}\n')
print(f'{"feed":<5}{"pool":>7}{"minL":>6}{"maxL":>6}{"meanL":>7}   ' + ''.join(f'{v:>10}' for v in VARIANTS))
for f in CATH_FEEDS:
    L = POOL_LEN[f]
    print(f'{f:<5}{len(L):>7}{L.min():>6}{L.max():>6}{L.mean():>7.1f}   '
          + ''.join(f'{len(VAR_IDX[f][v]):>10}' for v in VARIANTS))
for f in CATH_FEEDS:
    L = POOL_LEN[f]
    print(f'  {f}: shorter than K=16: {(L<16).sum()}, shorter than K=32: {(L<32).sum()} '
          f'(adaptive pooling repeats positions for these, as PyTorch does)')
print('\ndeployed - strict = the two RESCUED domains (lengths 34, 43). Expect exactly +2 per feed.')
for f in CATH_FEEDS:
    d, st = len(VAR_IDX[f]['deployed']), len(VAR_IDX[f]['strict'])
    assert d - st == 2, f'{f}: deployed-strict = {d-st}, expected 2 — check RESCUED / alphabet filter'
print('OK  deployed pools: ' + ', '.join(f'{f} {len(VAR_IDX[f]["deployed"])}' for f in CATH_FEEDS))

## 4. Oracle — built once on the unrestricted pool, subsets derived

The exhaustive all-pairs scan is the expensive step (~3.2x the filtered cost, roughly 13-16 min per feed).
It runs **once per feed on the unrestricted pool** and is cached to disk; the variant oracles are derived by
index restriction. Only the high-similarity pairs are stored — the full score matrix is never materialised.

In [ ]:
import hashlib
def pool_fingerprint(feed):
    """Cache key: the exact sequences, in order, plus the threshold. A cache built for a different
    pool, order, alphabet rule or BAND_HIGH must never be reused silently."""
    h = hashlib.sha1()
    for sq in POOL_SEQ[feed]: h.update(sq.encode()); h.update(b'\x00')
    h.update(f'|band={BAND_HIGH}|n={len(POOL_SEQ[feed])}'.encode())
    return h.hexdigest()[:16]

def build_oracle_full(feed, block=1024):
    fp = pool_fingerprint(feed)
    cf = os.path.join(CACHE_DIR, f'oracle_{feed}_{fp}.npz')
    if os.path.exists(cf):
        z = np.load(cf)
        if str(z['fingerprint']) == fp:
            print(f'  {feed}: loaded cached oracle ({len(z["pi"]):,} high pairs, fp={fp})')
            return z['pi'], z['pj'], z['ps']
        print(f'  {feed}: cache fingerprint mismatch — rebuilding')
    seqs = POOL_SEQ[feed]; lens = POOL_LEN[feed]; N = len(seqs)
    PI, PJ, PS = [], [], []
    t0 = time.time()
    for r0 in range(0, N, block):
        r1 = min(r0 + block, N)
        Dm = rf_cdist(seqs[r0:r1], seqs, scorer=RFLev.distance, workers=-1).astype(np.float64)
        den = np.maximum(lens[r0:r1][:, None], lens[None, :]); den[den == 0] = 1
        sim = 1.0 - Dm / den
        for a in range(r1 - r0):
            i = r0 + a; row = sim[a]
            hi = np.where(row >= BAND_HIGH)[0]; hi = hi[hi > i]
            if hi.size:
                PI.append(np.full(hi.size, i, dtype=np.int32)); PJ.append(hi.astype(np.int32))
                PS.append(row[hi].astype(np.float32))
        if r0 % (block * 4) == 0:
            print(f'    {feed} {r1}/{N}  [{time.time()-t0:.0f}s]', flush=True)
    pi = np.concatenate(PI) if PI else np.zeros(0, np.int32)
    pj = np.concatenate(PJ) if PJ else np.zeros(0, np.int32)
    ps = np.concatenate(PS) if PS else np.zeros(0, np.float32)
    np.savez_compressed(cf, pi=pi, pj=pj, ps=ps, fingerprint=fp)
    print(f'  {feed}: {len(pi):,} high pairs in {time.time()-t0:.0f}s -> cached')
    return pi, pj, ps

ORACLE_FULL = {}
for f in CATH_FEEDS:
    print(f'oracle {f} (unrestricted pool, N={len(POOL_SEQ[f])})...')
    ORACLE_FULL[f] = build_oracle_full(f)

In [ ]:
def derive(feed, keep):
    """Restrict the unrestricted oracle to a subset; returns (T_high, pos_pairs) in LOCAL indices."""
    N = len(POOL_SEQ[feed]); pi, pj, ps = ORACLE_FULL[feed]
    loc = np.full(N, -1, dtype=np.int64); loc[keep] = np.arange(len(keep))
    m = (loc[pi] >= 0) & (loc[pj] >= 0)
    a, b, s = loc[pi[m]], loc[pj[m]], ps[m]
    T = {}
    for x, y in ((a, b), (b, a)):
        order = np.argsort(x, kind='stable'); xs, ys = x[order], y[order]
        if xs.size:
            bounds = np.searchsorted(xs, np.unique(xs), side='left')
            uq = np.unique(xs); ends = np.append(bounds[1:], xs.size)
            for q, st, en in zip(uq, bounds, ends):
                T.setdefault(int(q), []).append(ys[st:en])
    T = {k: np.concatenate(v).astype(np.int32) for k, v in T.items()}
    return T, (a, b, s)

ORACLE = {}   # (feed, variant) -> dict(T_high=..., pos=(i,j,s))
for f in CATH_FEEDS:
    for v in VARIANTS:
        T, pos = derive(f, VAR_IDX[f][v])
        ORACLE[(f, v)] = dict(T_high=T, pos=pos)
        print(f'  {f:<4} {v:<7} pool={len(VAR_IDX[f][v]):>6}  queries@0.70={len(T):>6}  pairs={len(pos[0]):>8}')

## 5. Synthetic control feed (unchanged generator — isolates architecture from data)

In [ ]:
def build_synth_feed(n_perturb, n_indep, per_bin=STRAT_PER_BIN, seed=20260810, lo=50, hi=200):
    r = np.random.default_rng(seed); recs = []
    for _ in range(n_perturb):
        base = rand_seq(AA_ALPHABET, r, lo, hi)
        part = perturb(base, int(r.integers(0, len(base)+1)), AA_ALPHABET, r, hi)
        if 1 <= len(part) <= hi: recs.append((base, part))
    for _ in range(n_indep): recs.append((rand_seq(AA_ALPHABET, r, lo, hi), rand_seq(AA_ALPHABET, r, lo, hi)))
    recs = [(a, b, norm_lev(a, b)) for a, b in recs]; nl = np.array([x[2] for x in recs])
    bins = np.clip(np.digitize(nl, np.linspace(0, 1, 11)) - 1, 0, 9); take = []
    for bb in range(10):
        idx = np.where(bins == bb)[0]
        if idx.size: take.extend(r.permutation(idx)[:per_bin].tolist())
    seqs, I, J, NL = [], [], [], []
    for idx in take:
        a, b, l = recs[int(idx)]
        I.append(len(seqs)); seqs.append(a); J.append(len(seqs)); seqs.append(b); NL.append(l)
    return seqs, np.array(I), np.array(J), np.array(NL)

SYN_SEQ, SYN_I, SYN_J, SYN_NL = build_synth_feed(SYN_PERTURB, SYN_INDEP)
POOL_SEQ['synth'] = SYN_SEQ; POOL_LEN['synth'] = np.array([len(s) for s in SYN_SEQ])
VAR_IDX['synth'] = {'none': np.arange(len(SYN_SEQ))}

# synth oracle, built directly (small pool)
_lens = POOL_LEN['synth']; _N = len(SYN_SEQ); _PI, _PJ, _PS = [], [], []
for r0 in range(0, _N, 1024):
    r1 = min(r0 + 1024, _N)
    Dm = rf_cdist(SYN_SEQ[r0:r1], SYN_SEQ, scorer=RFLev.distance, workers=-1).astype(np.float64)
    den = np.maximum(_lens[r0:r1][:, None], _lens[None, :]); den[den == 0] = 1
    sim = 1.0 - Dm / den
    for a in range(r1 - r0):
        i = r0 + a; hi = np.where(sim[a] >= BAND_HIGH)[0]; hi = hi[hi > i]
        if hi.size:
            _PI.append(np.full(hi.size, i, np.int32)); _PJ.append(hi.astype(np.int32)); _PS.append(sim[a][hi].astype(np.float32))
ORACLE_FULL['synth'] = (np.concatenate(_PI) if _PI else np.zeros(0, np.int32),
                        np.concatenate(_PJ) if _PJ else np.zeros(0, np.int32),
                        np.concatenate(_PS) if _PS else np.zeros(0, np.float32))
T, pos = derive('synth', VAR_IDX['synth']['none'])
ORACLE[('synth', 'none')] = dict(T_high=T, pos=pos)
print(f"synth: pool={_N}  queries@0.70={len(T)}  (colab35 reference: 7296 / 2410)")

## 6. Evaluation pair sets, the null, and the audit

Two pair samples per (feed, variant):

- **decile-balanced** (`STRAT`) — 200,000 uniform candidate pairs plus all oracle pairs at >= 0.70, split
  into deciles by exact normLev, up to 400 per decile. This is what Spearman / AUROC are computed on, and
  it is *not* the natural pair distribution.
- **natural** (`DIST`) — uniform random pairs, used only for the score-distribution figure, together with a
  **null**: the identical index pairs scored on character-shuffled sequences. Shuffling preserves length and
  composition exactly and destroys order, so the gap between the observed and null curves is the similarity
  that is *not* explained by length and composition alone.

**Tie handling.** The length-ratio baseline produces enormous ties (every exact length twin scores 1.0), so
its MAP@10 is reported as a mean over **200** random tie-breaks (random subset *and* random order within a
tie group), with a 95% interval over repeats and the median tie-group size. On AA the query set is 10
(deployed) or 8 (strict), so that interval reflects tie-break variability only — the number remains an
anecdote for the same reason every AA retrieval number is. SNNEED scores are continuous cosines and do not
tie in practice.

In [ ]:
def build_strat(feed, variant, rng):
    keep = VAR_IDX[feed][variant]; seqs = POOL_SEQ[feed]; n = len(keep)
    a = rng.integers(0, n, STRAT_CAND); b = rng.integers(0, n, STRAT_CAND)
    m = a != b; a, b = a[m], b[m]
    nl = np.array([norm_lev(seqs[keep[i]], seqs[keep[j]]) for i, j in zip(a, b)])
    pi, pj, ps = ORACLE[(feed, variant)]['pos']
    if len(pi):
        a = np.concatenate([a, pi]); b = np.concatenate([b, pj]); nl = np.concatenate([nl, ps])
    bins = np.clip(np.digitize(nl, np.linspace(0, 1, 11)) - 1, 0, 9); ai, aj, av = [], [], []
    for bb in range(10):
        idx = np.where(bins == bb)[0]
        if idx.size:
            t = rng.permutation(idx)[:STRAT_PER_BIN]; ai.append(a[t]); aj.append(b[t]); av.append(nl[t])
    return dict(i=np.concatenate(ai).astype(np.int64), j=np.concatenate(aj).astype(np.int64),
                nl=np.concatenate(av))

STRAT = {}
for f in CATH_FEEDS:
    for v in VARIANTS:
        STRAT[(f, v)] = build_strat(f, v, np.random.default_rng(999))
        print(f'  strat {f:<4} {v:<7} n={len(STRAT[(f,v)]["nl"]):>5}')
STRAT[('synth', 'none')] = dict(i=SYN_I, j=SYN_J, nl=SYN_NL)
print(f'  strat synth  none    n={len(SYN_NL)}')

In [ ]:
def build_dist(feed, variant, rng):
    """Natural-pair sample + character-shuffled null on the same index pairs."""
    keep = VAR_IDX[feed][variant]; seqs = [POOL_SEQ[feed][k] for k in keep]; n = len(seqs)
    sh = [''.join(rng.permutation(list(s))) for s in seqs]
    a = rng.integers(0, n, NDIST); b = rng.integers(0, n, NDIST); m = a != b; a, b = a[m], b[m]
    obs = np.array([norm_lev(seqs[i], seqs[j]) for i, j in zip(a, b)])
    nul = np.array([norm_lev(sh[i],   sh[j])   for i, j in zip(a, b)])
    lr  = np.array([min(len(seqs[i]), len(seqs[j])) / max(len(seqs[i]), len(seqs[j])) for i, j in zip(a, b)])
    return dict(obs=obs, null=nul, lenratio=lr)

DIST = {}
t0 = time.time()
for f in CATH_FEEDS:
    for v in VARIANTS:
        DIST[(f, v)] = build_dist(f, v, np.random.default_rng(4242))
        d = DIST[(f, v)]
        print(f'  dist {f:<4} {v:<7} median obs={np.median(d["obs"]):.3f}  null={np.median(d["null"]):.3f}  '
              f'excess>null.p999={(d["obs"] > np.quantile(d["null"], 0.999)).mean()*100:.2f}%')
DIST[('synth', 'none')] = build_dist('synth', 'none', np.random.default_rng(4242))
print(f'  [{time.time()-t0:.0f}s]')

In [ ]:
KEYS = [('synth', 'none')] + [(f, v) for f in CATH_FEEDS for v in VARIANTS]
print(f'{"feed":<7}{"variant":<9}{"pool":>7}{"queries@.70":>13}{"oraclepairs":>13}'
      f'{"strat n":>9}{"hi":>6}{"far":>7}{"medObs":>8}{"medNull":>9}')
print('-' * 95)
AUDIT = {}
for f, v in KEYS:
    nl = STRAT[(f, v)]['nl']; d = DIST[(f, v)]
    row = dict(feed=f, variant=v, pool=len(VAR_IDX[f][v]), queries=len(ORACLE[(f, v)]['T_high']),
               oracle_pairs=int(len(ORACLE[(f, v)]['pos'][0])), strat_n=int(len(nl)),
               strat_high=int((nl >= BAND_HIGH).sum()), strat_far=int((nl < BAND_LOW).sum()),
               median_obs=float(np.median(d['obs'])), median_null=float(np.median(d['null'])))
    AUDIT[f'{f}|{v}'] = row
    print(f'{f:<7}{v:<9}{row["pool"]:>7}{row["queries"]:>13}{row["oracle_pairs"]:>13}'
          f'{row["strat_n"]:>9}{row["strat_high"]:>6}{row["strat_far"]:>7}'
          f'{row["median_obs"]:>8.3f}{row["median_null"]:>9.3f}')

EXPECT_DEPLOYED = {'3Di': (10501, 347), 'SS': (10497, 10002), 'AA': (10501, 10), 'synth': (7296, 2410)}
ALLOW_POOL_MISMATCH = False      # set True only if you have diffed and understand the difference
print('\nPOOL GATE — the deployed variant must reproduce colab35/colab34 exactly:')
_fail = []
for f in FEED_ORDER:
    v = 'none' if f == 'synth' else 'deployed'
    got = (AUDIT[f'{f}|{v}']['pool'], AUDIT[f'{f}|{v}']['queries']); exp = EXPECT_DEPLOYED[f]
    ok = got == exp
    print(f'  {f:<6}{v:<9} pool/queries got {got}  expected {exp}   {"OK" if ok else "*** MISMATCH"}')
    if not ok: _fail.append(f)
if _fail and not (QUICK or ALLOW_POOL_MISMATCH):
    raise AssertionError(f'deployed pool differs from colab35 on {_fail}. Stop and diff before running '
                         f'anything — every downstream comparison is anchored to this pool.')

for f, v in KEYS:
    q = AUDIT[f'{f}|{v}']['queries']
    if q == 0: print(f'  *** {f}/{v}: ZERO queries at >=0.70 — AUROC/MAP will be NaN.')
    elif q <= 20: print(f'  *** {f}/{v}: only {q} queries — AUROC/MAP/RMSE are anecdotes there.')
with open('colab36_audit.json','w') as fh: json.dump(AUDIT, fh, indent=2)

## 7. Metrics, and the length-ratio baseline

In [ ]:
def _auroc(sim, nl):
    y = (nl >= BAND_HIGH).astype(int)
    return roc_auc_score(y, sim) if 0 < y.sum() < len(y) else np.nan

def _rho(sim, nl):
    if len(nl) < 10 or np.ptp(nl) == 0: return np.nan
    r = spearmanr(sim, nl).correlation
    return float(r) if r == r else np.nan

BANDS = {'far': lambda nl: nl < BAND_LOW,
         'mid': lambda nl: (nl >= BAND_LOW) & (nl < BAND_HIGH),
         'high': lambda nl: nl >= BAND_HIGH}

def _ap_from_order(order, ts, k=10):
    hits = 0.0; ap = 0.0
    for rr, o in enumerate(order[:k], 1):
        if o in ts: hits += 1; ap += hits / rr
    return ap / min(len(ts), k)

def map10_emb(E_t, T_high, k=10, qb=256):
    q = list(T_high.keys())
    if not q: return np.nan
    aps = []
    for s0 in range(0, len(q), qb):
        qi = q[s0:s0+qb]; sc = E_t[qi] @ E_t.t()
        for r, idx in enumerate(qi): sc[r, idx] = -1e9
        top = torch.topk(sc, min(k, sc.shape[1]), dim=1).indices.cpu().numpy()
        for r, idx in enumerate(qi): aps.append(_ap_from_order(list(top[r]), set(T_high[idx].tolist()), k))
    return float(np.mean(aps))

def map10_length(lens, T_high, k=10, reps=200, seed=1234):
    """Retrieval using ONLY the length ratio min(La,Lb)/max(La,Lb).

    The score ties massively: every exact length twin scores 1.0, and the median query here has dozens of
    them, so the whole top-10 is usually a single tie group. A plain argpartition would report whichever
    tied element numpy happened to touch -- array order, not a measurement. We break ties uniformly at
    random (subset AND order within the group) and repeat `reps` times.

    Cheap because the ranking depends only on the QUERY LENGTH: candidates are grouped by score once per
    distinct length, and each repeat only has to sample the top k+1 out of the leading group(s).

    Returns (mean, sd, p2.5, p97.5, median tie-group size at k).
    """
    if not T_high: return (np.nan,) * 5
    L = lens.astype(np.float64); N = len(L); need = k + 1        # +1 so removing self still leaves k
    by_len = {}
    for qi in T_high: by_len.setdefault(int(L[qi]), []).append(qi)

    groups_by_len = {}; tie_sizes = []
    for lq in by_len:
        sc = np.minimum(L, lq) / np.maximum(L, lq)
        idx = np.argsort(-sc, kind='stable'); ss = sc[idx]
        bnd = np.flatnonzero(np.diff(ss)) + 1
        starts = np.concatenate([[0], bnd]); ends = np.concatenate([bnd, [N]])
        gs = []; tot = 0
        for a_, b_ in zip(starts, ends):
            gs.append(idx[a_:b_].copy()); tot += b_ - a_
            if tot >= need: break
        groups_by_len[lq] = gs
        kk = min(k, N - 1); thr = np.partition(sc, -kk)[-kk]
        tie_sizes.append(int((sc >= thr).sum()))

    rng = np.random.default_rng(seed); means = []
    for _ in range(reps):
        top_by_len = {}
        for lq, groups in groups_by_len.items():
            picked = []
            for g in groups:
                if len(picked) >= need: break
                m = need - len(picked)
                if len(g) <= m: picked.extend(g.tolist())
                else:
                    order = np.argsort(rng.random(len(g)))[:m]   # random subset AND random order
                    picked.extend(g[order].tolist())
            top_by_len[lq] = picked
        aps = []
        for lq, members in by_len.items():
            top = top_by_len[lq]
            for qi in members:
                aps.append(_ap_from_order([x for x in top if x != qi][:k],
                                          set(T_high[qi].tolist()), k))
        means.append(float(np.mean(aps)))
    means = np.asarray(means)
    return (float(means.mean()), float(means.std()),
            float(np.percentile(means, 2.5)), float(np.percentile(means, 97.5)),
            float(np.median(tie_sizes)))

def _record(method, feed, variant, sim, nl, map10, rmse_high=np.nan, seed=0, arm='', params=np.nan):
    rec = dict(method=method, arm=arm, feed=feed, variant=variant, seed=seed, params=params,
               map10_tiebreak_sd=np.nan, map10_lo=np.nan, map10_hi=np.nan,
               median_tie_group=np.nan,
               spearman=_rho(sim, nl), auroc=_auroc(sim, nl), map10=map10, rmse_high=rmse_high,
               n_pairs=int(len(nl)), n_queries=len(ORACLE[(feed, variant)]['T_high']))
    for bn, bf in BANDS.items():
        m = bf(nl); rec[f'spearman_{bn}'] = _rho(sim[m], nl[m]); rec[f'n_{bn}'] = int(m.sum())
    return rec

In [ ]:
# ---- length-ratio baseline: deterministic, no training, run once ----
len_rows = []
for f, v in KEYS:
    keep = VAR_IDX[f][v]; lens = POOL_LEN[f][keep]
    P = STRAT[(f, v)]; sim = np.minimum(lens[P['i']], lens[P['j']]) / np.maximum(lens[P['i']], lens[P['j']])
    m, sd, lo, hi, ties = map10_length(lens, ORACLE[(f, v)]['T_high'])
    rec = _record('Length', f, v, sim, P['nl'], m, arm='length-ratio')
    rec['map10_tiebreak_sd'] = sd; rec['map10_lo'] = lo; rec['map10_hi'] = hi
    rec['median_tie_group'] = ties
    len_rows.append(rec)
    nq = len(ORACLE[(f, v)]['T_high'])
    flag = '   <-- ANECDOTE: {} queries'.format(nq) if nq <= 20 else ''
    print(f'  Length {f:<6}{v:<9} rho={rec["spearman"]:+.3f}  AUROC={rec["auroc"]:.3f}  '
          f'MAP@10={m:.3f} sd={sd:.3f} [95% {lo:.3f}, {hi:.3f}]  median tie group {ties:.0f}{flag}')
print('\nRead every SNNEED number against these. Where SNNEED does not clearly beat Length,')
print('the metric is measuring length-matching, not edit distance.')
print('The interval is the spread over 200 random tie-breaks, NOT sampling error: where the median tie')
print('group is large, this baseline is "pick at random among length twins" and should be read as such.')
print('On feeds with <=20 queries (AA) the tie-break interval understates the real uncertainty, because')
print('the query set itself is tiny — quote it as an anecdote, exactly as for AA MAP@10 generally.')
print('\nNOTE for colab35: Dice on SS has the same pathology (19 distinct trigrams -> near-total set')
print('overlap -> huge ties). Its MAP@10 = 0.022 is arbitrary at the tie-break level. The conclusion')
print('("the ranking is noise") is unaffected, but the number should carry this caveat.')

## 8. Train and evaluate the six arms

In [ ]:
def build_pairs(n, seed, lo, hi):
    rng = np.random.default_rng(seed); pairs = []
    while len(pairs) < n:
        sd = rand_seq(AA_ALPHABET, rng, lo, hi); L = len(sd)
        t = float(rng.uniform(0, 1)); k = max(0, int(round((1-t)*L)))
        o = perturb(sd, k, AA_ALPHABET, rng, hi)
        if 1 <= len(o) <= hi: pairs.append((sd, o, norm_lev(sd, o)))
    return pairs

def make_collate(pad_w, truncate):
    def coll(batch):
        A = [x[0] for x in batch]; B = [x[1] for x in batch]; y = [x[2] for x in batch]
        # +2: conv1d(padding=1) applied twice reaches 2 positions past the sequence end. Padding to
        # exactly the batch maximum would let the tensor edge (implicit zeros) replace those pad
        # positions for the longest sequence only, making its embedding depend on its batch. See the
        # invariance check in section 2.
        w = pad_w or (max(max(len(a) for a in A), max(len(b) for b in B)) + 2)
        xa, la = encode_batch(A, w, truncate); xb, lb = encode_batch(B, w, truncate)
        return xa, la, xb, lb, torch.tensor(y, dtype=torch.float32)
    return coll

class PairDS(Dataset):
    def __init__(s, pp): s.p = pp
    def __len__(s): return len(s.p)
    def __getitem__(s, i): return s.p[i]

def train_arm(arm, seed):
    lo, hi = arm['train_len']
    pad_w = PAD_MAX if arm['pad_w'] == 'max' else arm['pad_w']
    pairs = build_pairs(N_TRAIN, seed, lo, hi)
    lab = np.array([l for *_, l in pairs])
    print(f"  pairs n={len(pairs)} median={np.median(lab):.3f} "
          f"far={int((lab<BAND_LOW).sum())} mid={int(((lab>=BAND_LOW)&(lab<BAND_HIGH)).sum())} "
          f"high={int((lab>=BAND_HIGH).sum())}")
    torch.manual_seed(seed)
    model = RegModel(EncPool(arm['K'], arm['c2ch'], arm['pool'])).to(device)
    dl = DataLoader(PairDS(pairs), batch_size=BS, shuffle=True,
                    collate_fn=make_collate(pad_w, arm['truncate']))
    opt = torch.optim.Adam(model.parameters(), 1e-3); model.train(); t0 = time.time()
    for ep in range(1, EPOCHS+1):
        tot = nb = 0
        for xa, la, xb, lb, y in dl:
            xa, la, xb, lb, y = xa.to(device), la.to(device), xb.to(device), lb.to(device), y.to(device)
            loss = ((model(xa, la, xb, lb) - y)**2).mean()
            opt.zero_grad(); loss.backward(); opt.step(); tot += loss.item(); nb += 1
        if ep % 10 == 0 or ep == 1: print(f'    epoch {ep:>2}/{EPOCHS}  MSE {tot/nb:.5f}', flush=True)
    if device.type == 'cuda': torch.cuda.synchronize()
    model.eval(); print(f'    trained in {time.time()-t0:.0f}s')
    return model

@torch.no_grad()
def embed_pool(model, arm, feed, bs=256):
    seqs = POOL_SEQ[feed]; pad_w = PAD_MAX if arm['pad_w'] == 'max' else arm['pad_w']
    order = np.argsort([len(s) for s in seqs]); out = [None]*len(seqs)
    for i in range(0, len(order), bs):
        idx = order[i:i+bs]; batch = [seqs[j] for j in idx]
        w = pad_w or (max(len(s) for s in batch) + 2)     # see make_collate
        x, l = encode_batch(batch, w, arm['truncate'])
        e = model.encoder(x.to(device), l.to(device)).cpu().numpy()
        for kk, j in enumerate(idx): out[j] = e[kk]
    return np.stack(out).astype(np.float32)

In [ ]:
def eval_model(model, arm, seed):
    out = []
    for f in ['synth'] + CATH_FEEDS:
        E_full = embed_pool(model, arm, f)
        vs = ['none'] if f == 'synth' else VARIANTS
        for v in vs:
            keep = VAR_IDX[f][v]; E = E_full[keep]; P = STRAT[(f, v)]
            sim  = np.sum(E[P['i']] * E[P['j']], axis=1)
            pred = 1.0 - np.linalg.norm(E[P['i']] - E[P['j']], axis=1) / 2.0
            hm = P['nl'] >= BAND_HIGH
            rmse = float(np.sqrt(np.mean((pred[hm] - P['nl'][hm])**2))) if hm.sum() else np.nan
            m10 = map10_emb(torch.as_tensor(E, device=device), ORACLE[(f, v)]['T_high'])
            out.append(_record('SNNEED', f, v, sim, P['nl'], m10, rmse, seed,
                               arm=arm['name'], params=arm['params']))
        got = {r['variant']: r['spearman'] for r in out[-len(vs):]}
        print(f"    {f:<6} " + '  '.join(f'{k}:rho={x:+.2f}' for k, x in got.items()), flush=True)
    return out

# ---------------- GATE: train A0 seed 0 first, check against colab35, only then continue ----------
REF35 = {'synth': (0.926, 0.972), '3Di': (0.953, 0.515), 'SS': (0.963, 0.405), 'AA': (0.183, 0.928)}
TOL   = {'synth': (0.06, 0.06), '3Di': (0.07, 0.10), 'SS': (0.06, 0.08), 'AA': (0.20, 0.25)}
ALLOW_GATE_FAIL = False

A0 = ARMS[0]; assert A0['name'] == 'A0 deployed'
print(f"\n=========== GATE: {A0['name']}  seed {SEEDS[0]} ===========", flush=True)
_m = train_arm(A0, SEEDS[0]); rows = list(len_rows) + eval_model(_m, A0, SEEDS[0])
del _m
if device.type == 'cuda': torch.cuda.empty_cache()

print('\nREPLICATION GATE (A0, deployed variant, single seed vs the colab35 3-seed mean):')
_bad = []
for f in FEED_ORDER:
    v = 'none' if f == 'synth' else 'deployed'
    r = [x for x in rows if x['method'] == 'SNNEED' and x['feed'] == f and x['variant'] == v][0]
    dr = abs(r['spearman'] - REF35[f][0]); dm = abs(r['map10'] - REF35[f][1])
    ok = (dr <= TOL[f][0]) and (np.isnan(dm) or dm <= TOL[f][1])
    print(f'  {f:<6} rho {r["spearman"]:+.3f} vs {REF35[f][0]:+.3f} (d={dr:.3f}, tol {TOL[f][0]}) | '
          f'MAP {r["map10"]:.3f} vs {REF35[f][1]:.3f} (d={dm:.3f}, tol {TOL[f][1]})  '
          f'{"OK" if ok else "*** OUT OF TOLERANCE"}')
    if not ok: _bad.append(f)
if _bad and not (QUICK or ALLOW_GATE_FAIL):
    raise AssertionError(f'A0 failed to reproduce colab35 on {_bad}. Every arm comparison below is '
                         f'anchored to A0 — fix this before spending GPU time on A1-A5.')
print('GATE PASSED — proceeding to the remaining arms.' if not _bad else 'GATE BYPASSED (QUICK/override).')

for arm in ARMS:
    for seed in SEEDS:
        if arm['name'] == A0['name'] and seed == SEEDS[0]: continue      # already run as the gate
        print(f"\n=========== {arm['name']}  seed {seed} ===========", flush=True)
        model = train_arm(arm, seed); rows += eval_model(model, arm, seed)
        del model
        if device.type == 'cuda': torch.cuda.empty_cache()

df = pd.DataFrame(rows); df.to_csv('colab36_metrics.csv', index=False)
print(f'\nsaved colab36_metrics.csv ({len(df)} rows)')

## 9. Results

In [ ]:
agg = (df.groupby(['method','arm','feed','variant'], dropna=False)
         .agg(spearman=('spearman','mean'), sp_sd=('spearman','std'),
              sp_far=('spearman_far','mean'), sp_high=('spearman_high','mean'),
              auroc=('auroc','mean'), map10=('map10','mean'), map_sd=('map10','std'),
              rmse_high=('rmse_high','mean'), params=('params','first'),
              map10_tiebreak_sd=('map10_tiebreak_sd','first'),
              map10_lo=('map10_lo','first'), map10_hi=('map10_hi','first'),
              median_tie_group=('median_tie_group','first'),
              n_pairs=('n_pairs','first'), n_queries=('n_queries','first')).reset_index())
agg.to_csv('colab36_summary.csv', index=False)
pd.set_option('display.width', 240, 'display.max_columns', 40, 'display.max_rows', 200)

print('=== REPLICATION CHECK: A0 on the 50-200 variant vs colab35 ===')
ref = {'synth': (0.926, 0.972), '3Di': (0.953, 0.515), 'SS': (0.963, 0.405), 'AA': (0.183, 0.928)}
for f in FEED_ORDER:
    v = 'none' if f == 'synth' else 'deployed'
    r = agg[(agg.arm == 'A0 deployed') & (agg.feed == f) & (agg.variant == v)]
    if len(r):
        r = r.iloc[0]
        print(f'  {f:<6} rho {r.spearman:+.3f} (colab35 {ref[f][0]:+.3f})   '
              f'MAP@10 {r.map10:.3f} (colab35 {ref[f][1]:.3f})')

In [ ]:
ARM_ORDER = [a['name'] for a in ARMS] + ['length-ratio']
for m, name in [('spearman','Spearman rho'), ('map10','MAP@10'), ('sp_high','Spearman, high band')]:
    print(f'\n================ {name} ================')
    for f in FEED_ORDER:
        vs = ['none'] if f == 'synth' else VARIANTS
        sub = agg[(agg.feed == f) & agg[m].notna()]
        if not len(sub): print(f'\n-- {f} -- (no values: all NaN)'); continue
        t = sub.pivot_table(index='arm', columns='variant', values=m)
        t = t.reindex(index=[a for a in ARM_ORDER if a in t.index],
                      columns=[c for c in vs if c in t.columns])
        print(f'\n-- {f} --'); print(t.to_string(float_format=lambda x: f'{x:7.3f}'))

In [ ]:
# PRIMARY analysis: one arm, designated before the run. Best-of-six is reported separately and is
# exploratory — it selects on the same evaluation data, which is optimistic, badly so for AA (10 queries).
print(f'=== PRIMARY: {PRIMARY_ARM} vs the length-ratio baseline (MAP@10) ===')
for f in FEED_ORDER:
    for v in (['none'] if f == 'synth' else VARIANTS):
        pr = agg[(agg.arm==PRIMARY_ARM) & (agg.feed==f) & (agg.variant==v)].dropna(subset=['map10'])
        lb = agg[(agg.method=='Length') & (agg.feed==f) & (agg.variant==v)].dropna(subset=['map10'])
        if not len(pr) or not len(lb):
            print(f'  {f:<6}{v:<9} (no MAP@10 — no queries at >=0.70)'); continue
        pr, lb = pr.iloc[0], lb.iloc[0]
        print(f'  {f:<6}{v:<9} {PRIMARY_ARM} {pr.map10:.3f} (sd {pr.map_sd:.3f})  '
              f'Length {lb.map10:.3f} [95% {lb.map10_lo:.3f}, {lb.map10_hi:.3f}], median tie group '
              f'{lb.median_tie_group:.0f}  delta {pr.map10 - lb.map10:+.3f}')

print('\n=== EXPLORATORY: best of six arms per cell (selected on the evaluation data — optimistic) ===')
for f in FEED_ORDER:
    for v in (['none'] if f == 'synth' else VARIANTS):
        snn = agg[(agg.method=='SNNEED') & (agg.feed==f) & (agg.variant==v)].dropna(subset=['map10'])
        if not len(snn): continue
        best = snn.loc[snn.map10.idxmax()]
        pr = agg[(agg.arm==PRIMARY_ARM) & (agg.feed==f) & (agg.variant==v)].dropna(subset=['map10'])
        gap = f'{best.map10 - pr.iloc[0].map10:+.3f} over primary' if len(pr) else 'n/a'
        print(f'  {f:<6}{v:<9} best={best.arm:<22} MAP {best.map10:.3f}   ({gap})')
print('\nTreat the exploratory row as a hypothesis for a fresh run, never as the reported result —')
print('especially on AA, where MAP@10 rests on 10 queries.')

In [ ]:
import matplotlib.pyplot as plt
FEED_C = {'synth':'#E8871A', '3Di':'#2E6DB4', 'SS':'#C0392B', 'AA':'#8A8F98'}

fig, axes = plt.subplots(len(CATH_FEEDS), len(VARIANTS), figsize=(15, 10), sharex=True)
bins = np.linspace(0, 1, 81); ctr = 0.5*(bins[1:] + bins[:-1])
for r, f in enumerate(CATH_FEEDS):
    for c, v in enumerate(VARIANTS):
        ax = axes[r, c]; d = DIST[(f, v)]
        for key, lab, ls, alpha in [('obs', 'observed', '-', 0.25), ('null', 'null (shuffled)', '--', 0.0)]:
            h, _ = np.histogram(d[key], bins=bins, density=True)
            ax.plot(ctr, h, ls, color=FEED_C[f], lw=2 if key == 'obs' else 1.4,
                    alpha=1.0 if key == 'obs' else 0.75, label=lab)
            if alpha: ax.fill_between(ctr, h, color=FEED_C[f], alpha=alpha)
        ax.axvline(BAND_HIGH, color='grey', ls=':', lw=1)
        ax.set_title(f'{f} — {v}  (median obs {np.median(d["obs"]):.2f} / null {np.median(d["null"]):.2f})',
                     fontsize=10)
        ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
        if r == len(CATH_FEEDS)-1: ax.set_xlabel('normalized Levenshtein similarity (true score)')
        if c == 0: ax.set_ylabel('density')
        if r == 0 and c == 0: ax.legend(fontsize=8, frameon=False)
plt.tight_layout(); plt.savefig('colab36_score_distributions.png', dpi=150, bbox_inches='tight'); plt.show()

In [ ]:
# Slide figure: observed vs null for the unrestricted pool, all feeds on one axis.
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
for a, v, ttl in [(ax[0], 'deployed', 'current filter [50, 200] + RESCUED'),
                  (ax[1], 'none', 'no length filter')]:
    for f in CATH_FEEDS + ['synth']:
        key = (f, 'none') if f == 'synth' else (f, v)
        if key not in DIST: continue
        d = DIST[key]
        h, _ = np.histogram(d['obs'], bins=bins, density=True)
        a.plot(ctr, h, '-' if f != 'synth' else '--', color=FEED_C[f], lw=2,
               label=f'{f} (median {np.median(d["obs"]):.2f})')
        hn, _ = np.histogram(d['null'], bins=bins, density=True)
        a.plot(ctr, hn, ':', color=FEED_C[f], lw=1.2, alpha=0.8)
    a.axvline(BAND_HIGH, color='grey', ls=':', lw=1); a.set_title(ttl)
    a.set_xlabel('normalized Levenshtein similarity'); a.legend(fontsize=8, frameon=False)
    a.spines['top'].set_visible(False); a.spines['right'].set_visible(False)
ax[0].set_ylabel('density')
plt.suptitle('Observed (solid) vs character-shuffled null (dotted) — the gap is the retrievable signal')
plt.tight_layout(); plt.savefig('colab36_null_overlay.png', dpi=150, bbox_inches='tight'); plt.show()

## 10. How to read this notebook

1. **Replication first.** If `A0` on `50-200` does not reproduce colab35, nothing below is interpretable.
2. **A1 vs A0** tests the zero-bucket mechanism. A1 should degrade badly. If it does not, the claim that
   fixed-width pooling encodes length by bucket occupancy is wrong and the Methods text (patch P3) needs
   revisiting.
3. **A2 vs A0 on `50-200`** is architecture at fixed data. If A2 is *worse*, true-length pooling discarded a
   length cue that was genuinely predictive — the follow-up is an explicit length feature, not a return to
   fixed pooling.
4. **`none` vs `50-200` within an arm** is the actual question: does the filter buy anything?
5. **Always against `Length`.** Any cell where SNNEED does not clearly beat the length-ratio baseline is
   measuring length-matching. Expect this to bite hardest on the unrestricted variants.
6. **The null curves** are the honest replacement for the Tracy-Widom inset on slide 19: the floor is
   measured per alphabet from the data itself, and the observed-minus-null gap is the signal any method
   could find. No Tracy-Widom claim is made or needed.

**Known limitations, stated rather than hidden**

- Training batches are randomly shuffled, **not** length-bucketed. That is deliberate — bucketing would
  correlate batch composition with length and differ in effect across arms — but it means A5 (train 20-800)
  pads most batches near 800 and costs roughly 4x A2's training time. Budget ~3 min per A5 seed.
- **A4 is matched-parameter, not a clean resolution isolation**: it moves K 16->32 and conv2 64->32 together.
- The primary comparison is the pre-designated `A2 true K16`; the best-of-six table is exploratory.
- `QUICK = True` subsamples every pool to ~2,000 sequences so the exhaustive oracles are cheap. It is a
  smoke pass; its numbers are not results and both gates are bypassed.

Outputs: `colab36_metrics.csv`, `colab36_summary.csv`, `colab36_audit.json`, `environment_colab36.json`,
`colab36_score_distributions.png`, `colab36_null_overlay.png`.